In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

# =========================================================
# LOAD DATA
# =========================================================

umap_embeddings = np.load(
    "../embeddings/umap_embeddings.npy"
)

cluster_labels = np.load(
    "../embeddings/hdbscan_labels.npy"
)

paths = np.load(
    "../embeddings/image_paths.npy",
    allow_pickle=True
)

# =========================================================
# CREATE DATAFRAME
# =========================================================

df = pd.DataFrame({

    "x": umap_embeddings[:,0],

    "y": umap_embeddings[:,1],

    "cluster": cluster_labels.astype(str),

    "path": paths
})

print(df.head())

# =========================================================
# INTERACTIVE UMAP PLOT
# =========================================================

fig = px.scatter(

    df,

    x="x",

    y="y",

    color="cluster",

    hover_data=["path"],

    title="Interactive UMAP of Scientific Images",

    width=1000,

    height=800
)

fig.update_traces(

    marker=dict(size=5)
)

fig.show()

          x         y cluster                                 path
0 -0.374808  5.982678       8  ..\data\resized\img10000_single.jpg
1 -0.439833  5.854743       8  ..\data\resized\img10001_single.jpg
2 -0.079611  5.407290       8       ..\data\resized\img10002_A.jpg
3  0.289189  5.375375       8       ..\data\resized\img10002_B.jpg
4  0.055314  5.344043       8       ..\data\resized\img10002_C.jpg


thumbnail umap

In [2]:
import numpy as np
import pandas as pd

from PIL import Image
from pathlib import Path

import plotly.express as px
import plotly.io as pio

import base64
from io import BytesIO

# =========================================================
# RENDERER
# =========================================================

pio.renderers.default = "browser"

# =========================================================
# LOAD DATA
# =========================================================

umap_embeddings = np.load(
    "../embeddings/umap_embeddings.npy"
)

cluster_labels = np.load(
    "../embeddings/hdbscan_labels.npy"
)

paths = np.load(
    "../embeddings/image_paths.npy",
    allow_pickle=True
)

# =========================================================
# CREATE THUMBNAILS
# =========================================================

def image_to_base64(path, size=(128,128)):

    try:

        img = Image.open(path).convert("RGB")

        img.thumbnail(size)

        buffer = BytesIO()

        img.save(buffer, format="PNG")

        encoded = base64.b64encode(
            buffer.getvalue()
        ).decode()

        return (
            f"<img src='data:image/png;base64,"
            f"{encoded}'>"
        )

    except Exception as e:

        return "Image Error"

# =========================================================
# CREATE IMAGE HTML
# =========================================================

thumbnail_html = []

print("Generating thumbnails...")

for path in paths:

    thumbnail_html.append(
        image_to_base64(path)
    )

# =========================================================
# DATAFRAME
# =========================================================

df = pd.DataFrame({

    "x": umap_embeddings[:,0],

    "y": umap_embeddings[:,1],

    "cluster": cluster_labels.astype(str),

    "path": paths,

    "thumbnail": thumbnail_html
})

# =========================================================
# INTERACTIVE PLOT
# =========================================================

fig = px.scatter(

    df,

    x="x",

    y="y",

    color="cluster",

    hover_data=["path"],

    width=1200,

    height=900,

    title="Interactive Scientific Image Explorer"
)

# =========================================================
# CUSTOM HOVER TEMPLATE
# =========================================================

fig.update_traces(

    marker=dict(size=5),

    hovertemplate=
    "<b>Cluster:</b> %{marker.color}<br><br>" +
    "%{customdata[1]}<br><br>" +
    "%{customdata[0]}<extra></extra>",

    customdata=np.stack(
        (
            df["thumbnail"],
            df["path"]
        ),
        axis=-1
    )
)

# =========================================================
# SHOW
# =========================================================

fig.show()

Generating thumbnails...


KeyboardInterrupt: 